# Step 10a – Classified maps from the ten tuned models

Paper 2: *Benchmarking Ten Machine-Learning Algorithms for Mediterranean Forest Mapping with Sentinel-2* (Remote Sensing, 2026).

- **In the paper:** Sections 2.5, 3.5 · Table 9, Table S4
- **Needs:** step 04 (`RESULTS_DIR/train_test_gap/*.joblib`), step 03 (`RESULTS_DIR/correlation_training_only/`), the sample table and the stack tiles `P2_stack_10m_part*.tif` from `gee/02_predictor_stack_and_exports.js` in `DATA_DIR`
- **Writes:** `DATA_DIR/maps/<model>/*.tif` (used by steps 10b–13), plus the consistency check, class areas and prediction times in `DATA_DIR/maps/`

The run first checks that the stack matches the training table (all 16 predictors must correlate at r ≥ 0.9) and stops otherwise. Prediction takes several hours; finished tiles are skipped when the notebook is run again.

Run the configuration cell below first.

In [ ]:
# ---------------------------------------------------------------- paths
# The data are not included in this repository. Set the two folders below to
# where the files are on your computer (or on your Google Drive in Colab).
#   DATA_DIR    : input data (training samples, Sentinel-2 stack, map tiles, rasters)
#   RESULTS_DIR : outputs written by the notebooks. Later steps read the outputs
#                 of earlier ones (e.g. RESULTS_DIR/train_test_gap/ from step 04).
import os
DATA_DIR    = 'data/'
RESULTS_DIR = 'results/'

# Google Colab: mount Drive and point the folders to it, for example
# from google.colab import drive; drive.mount('/content/drive')
# DATA_DIR    = '/content/drive/MyDrive/<your_folder>/'
# RESULTS_DIR = '/content/drive/MyDrive/<your_folder>/results/'

os.makedirs(RESULTS_DIR, exist_ok=True)

# Files and folders used by this notebook:
#   DATA_DIR / S2_Features_For_Correlation_v3.csv
#   DATA_DIR / P2_stack_10m_part*.tif
#   DATA_DIR / maps/
#   RESULTS_DIR / correlation_training_only/
#   RESULTS_DIR / train_test_gap/

In [ ]:
!pip install -q catboost xgboost rasterio pyproj

In [ ]:
# Optional (Google Colab): copy the stack tiles (~20 GB) from Drive to the local disk first.
# Reading them locally makes the prediction much faster. Skip this cell when running locally.
!df -h .
!mkdir -p stack && cp {DATA_DIR}P2_stack_10m_part*.tif stack/
!ls -lh stack
STACK_DIR = 'stack/'

In [ ]:
"""
Classified maps from all ten models

Run AFTER step 04 (uses the ten tuned models saved there) and after exporting the
16-band predictor stack with gee/02_predictor_stack_and_exports.js.

Steps:
  1. Consistency check: samples the exported stack at the sample points (2,800 in the paper) and
     compares with the CSV used for training (same predictors, same scaling?).
  2. Block-wise prediction of every tile with every model -> uint8 GeoTIFFs
     (resumable: finished tiles are skipped, so a Colab disconnect loses little).
  3. Mapped area per class and model (km2 and %).
  KNN and SVM-RBF (very slow to predict) are mapped on a systematic sample of the 10 m
  pixels (centre pixel of every 3 x 3 block = 30 m map); see STRIDE.

Outputs (OUT_DIR):
  <model>/<tile>_<model>.tif          classified tiles (class codes, 255 = no data)
  stack_consistency_check.csv
  class_areas_km2.csv, class_areas_pct.csv
  map_prediction_times.csv
"""

import os, glob, json, time
import numpy as np
import pandas as pd
import joblib
import rasterio
from rasterio.windows import Window
from pyproj import Transformer
from sklearn.preprocessing import LabelEncoder

# ------------------------------------------------------------------ config
DATA_PATH  = DATA_DIR + 'S2_Features_For_Correlation_v3.csv'
SCREEN_DIR = RESULTS_DIR + 'correlation_training_only/'
MODEL_DIR  = RESULTS_DIR + 'train_test_gap/'
STACK_DIR  = STACK_DIR if 'STACK_DIR' in globals() else DATA_DIR   # folder with the P2_stack_10m_part*.tif tiles
OUT_DIR    = DATA_DIR + 'maps/'   # read by steps 10b-13
LABEL_COL  = 'class'
NODATA_IN  = -9999          # value set with unmask()/noData in the GEE export
NODATA_OUT = 255
BLOCK      = 1023           # pixels per block side (multiple of 3); lower it if Colab runs out of RAM
# Slow models (every pixel compared with training points / support vectors) are predicted on a
# systematic sample: the centre 10 m pixel of every 3 x 3 block, written as a 30 m map.
# Set a model to 1 for full 10 m. Areas are scaled accordingly (unbiased systematic sample).
STRIDE     = {'KNN': 3, 'SVM-RBF': 3}
# Fast models first; KNN and SVM are by far the slowest to predict.
MODELS = ['CatBoost', 'XGBoost', 'GTB', 'RF', 'CART', 'NB', 'MDC', 'MLP', 'KNN', 'SVM-RBF']

os.makedirs(OUT_DIR, exist_ok=True)
PREDICTORS = pd.read_csv(SCREEN_DIR + 'v5_final_vif_training.csv')['feature'].tolist()  # model order
# Band order of the GEE export = the PREDICTORS list in gee/02_predictor_stack_and_exports.js. This is NOT the order
# of the VIF file above (sorted by VIF); bands are renamed with STACK_BANDS and then
# re-ordered to the model order before every prediction.
STACK_BANDS = ['slope', 'elevation', 'NDWI', 'SATVI', 'NDGR', 'B8A', 'B2',
               'NDVI_idm_7x7', 'NDVI_svar_7x7', 'NDVI_corr_7x7', 'NDVI_shade_7x7', 'NDVI_prom_7x7',
               'NDVI_diss_11x11', 'NDVI_imcorr2_11x11', 'NDVI_corr_11x11', 'NDVI_shade_11x11']
assert sorted(STACK_BANDS) == sorted(PREDICTORS), 'STACK_BANDS and the VIF file list differ'
df = pd.read_csv(DATA_PATH)
codes = LabelEncoder().fit(df[LABEL_COL]).classes_     # model output k -> original class code
assert np.issubdtype(codes.dtype, np.integer) and codes.max() < NODATA_OUT, \
    'Class codes must be integers 0-254 to be written as uint8'

tiles = sorted(glob.glob(STACK_DIR + 'P2_stack_10m_part*.tif'))
assert tiles, f'No .tif files in {STACK_DIR}'
with rasterio.open(tiles[0]) as src:
    CRS, px = src.crs, abs(src.transform.a * src.transform.e)
    assert src.count == len(STACK_BANDS), f'Stack has {src.count} bands, expected {len(STACK_BANDS)}'
    assert CRS.is_projected, 'Export the stack in a projected CRS (e.g. UTM) so areas are in m2'
print(f'{len(tiles)} tile(s) | CRS {CRS} | pixel {np.sqrt(px):.1f} m')
print('Raster band order (GEE export):', STACK_BANDS)
print('Model feature order (VIF file):  ', PREDICTORS)

# ------------------------------------------------------------------ 1. consistency check
def get_lonlat(frame):
    if '.geo' in frame.columns:
        return np.array([json.loads(g)['coordinates'] for g in frame['.geo']], dtype=float)
    for lon, lat in [('longitude', 'latitude'), ('lon', 'lat'), ('Longitude', 'Latitude')]:
        if lon in frame.columns and lat in frame.columns:
            return frame[[lon, lat]].to_numpy(dtype=float)
    raise ValueError(f'No coordinates found. Columns: {frame.columns.tolist()}')

lonlat = get_lonlat(df)
xs, ys = Transformer.from_crs('EPSG:4326', CRS, always_xy=True).transform(lonlat[:, 0], lonlat[:, 1])
vals = np.full((len(df), len(STACK_BANDS)), np.nan)
for t in tiles:
    with rasterio.open(t) as src:
        b = src.bounds
        todo = (xs >= b.left) & (xs < b.right) & (ys > b.bottom) & (ys <= b.top) & np.isnan(vals[:, 0])
        if todo.any():
            vals[todo] = np.array(list(src.sample(zip(xs[todo], ys[todo]))), dtype=float)
vals[vals == NODATA_IN] = np.nan
vals = pd.DataFrame(vals, columns=STACK_BANDS)[PREDICTORS].to_numpy()   # raster -> model order
ref = df[PREDICTORS].to_numpy(dtype=float)
ok = ~np.isnan(vals).any(1)
check = pd.DataFrame({
    'predictor': PREDICTORS,
    'pearson_r': [np.corrcoef(vals[ok, j], ref[ok, j])[0, 1] for j in range(len(PREDICTORS))],
    'median_abs_diff': np.nanmedian(np.abs(vals[ok] - ref[ok]), axis=0),
    'csv_median': np.median(ref[ok], axis=0),
    'raster_median': np.median(vals[ok], axis=0)}).round(4)
check.to_csv(OUT_DIR + 'stack_consistency_check.csv', index=False)
print(f'\nReference points inside the stack: {ok.sum()} of {len(df)}')
print(check.to_string(index=False))
low = check[check['pearson_r'] < 0.9]
if len(low):   # hard stop: never predict from a stack that does not match the training data
    raise RuntimeError(f'Predictors do not match the training CSV (r < 0.9): {low["predictor"].tolist()}. '
                       'Check band order, scaling and texture settings. No maps were predicted.')
print('\nConsistency check passed: all predictors match the training data.')

# ------------------------------------------------------------------ 2. prediction
def predict_tile(model, src_path, out_path, stride=1):
    assert BLOCK % 3 == 0 and BLOCK % stride == 0
    off = stride // 2                                   # centre pixel of each stride x stride cell
    with rasterio.open(src_path) as src:
        prof = src.profile.copy()
        out_h, out_w = len(range(off, src.height, stride)), len(range(off, src.width, stride))
        prof.update(driver='GTiff', count=1, dtype='uint8', nodata=NODATA_OUT, compress='deflate',
                    tiled=True, blockxsize=256, blockysize=256, BIGTIFF='IF_SAFER',
                    height=out_h, width=out_w,
                    transform=src.transform * src.transform.scale(stride, stride))
        with rasterio.open(out_path + '.part', 'w', **prof) as dst:
            for row in range(0, src.height, BLOCK):
                for col in range(0, src.width, BLOCK):
                    win = Window(col, row, min(BLOCK, src.width - col), min(BLOCK, src.height - row))
                    arr = src.read(window=win).astype('float32')[:, off::stride, off::stride]
                    h, w = arr.shape[1:]
                    if h == 0 or w == 0:
                        continue
                    win_out = Window(col // stride, row // stride, w, h)
                    X = arr.reshape(len(STACK_BANDS), -1).T
                    valid = np.isfinite(X).all(1) & (X != NODATA_IN).all(1)
                    out = np.full(h * w, NODATA_OUT, dtype=np.uint8)
                    if valid.any():
                        Xv = pd.DataFrame(X[valid], columns=STACK_BANDS)[PREDICTORS]   # model order
                        k = np.ravel(model.predict(Xv)).astype(int)
                        out[valid] = codes[k]
                    dst.write(out.reshape(h, w), 1, window=win_out)
    os.replace(out_path + '.part', out_path)

def class_counts(path):
    counts = np.zeros(256, dtype=np.int64)
    with rasterio.open(path) as src:
        for _, win in src.block_windows(1):
            counts += np.bincount(src.read(1, window=win).ravel(), minlength=256)
    return counts

times, areas = [], {}
for m in MODELS:
    model = joblib.load(MODEL_DIR + f'{m}.joblib')
    stride = STRIDE.get(m, 1)
    out_m = OUT_DIR + f'{m}/'
    os.makedirs(out_m, exist_ok=True)
    t0 = time.perf_counter()
    counts = np.zeros(256, dtype=np.int64)
    for t in tiles:
        out_path = out_m + os.path.basename(t).replace('.tif', f'_{m}.tif')
        if not os.path.exists(out_path):
            print(f'{m}: predicting {os.path.basename(t)} ({10 * stride} m) ...')
            predict_tile(model, t, out_path, stride)
    # count over ALL finished tiles of this model, so the areas stay complete when the
    # stack is exported and processed in batches (predictor tiles deleted after use)
    for f in sorted(glob.glob(out_m + '*.tif')):
        counts += class_counts(f)
    elapsed = time.perf_counter() - t0
    times.append({'Model': m, 'map_resolution_m': round(np.sqrt(px)) * stride,
                  'predict_time_min': round(elapsed / 60, 1)})
    areas[m] = {c: counts[c] * px * stride ** 2 / 1e6 for c in codes}   # km2
    print(f'{m}: done in {elapsed/60:.1f} min | area (km2):',
          {int(c): round(a, 1) for c, a in areas[m].items()})

area_km2 = pd.DataFrame(areas).T.round(2)
area_km2.columns = [f'class_{c}' for c in codes]
area_km2.to_csv(OUT_DIR + 'class_areas_km2.csv')
(100 * area_km2.div(area_km2.sum(1), axis=0)).round(2).to_csv(OUT_DIR + 'class_areas_pct.csv')
pd.DataFrame(times).to_csv(OUT_DIR + 'map_prediction_times.csv', index=False)
print('\n', area_km2.to_string())
print('\nFiles saved to:', OUT_DIR)